# 03 — Avaliação Comparativa TF-IDF vs BERTimbau
**NurseXAI** | Métricas, Bootstrap CI e relatório por label

---
**Requer:** `02_model_training.ipynb` executado

Este notebook realiza:
1. TF-IDF baseline no mesmo holdout do BERT
2. Extracção de probabilidades BERTimbau
3. Comparação com Bootstrap CI 95%
4. Tabela e relatório por label para o artigo

## 1. Setup

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
from functools import partial
from sklearn.utils import resample
from sklearn.metrics import f1_score, hamming_loss, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

df         = pd.read_csv('../data/notas_processadas.csv', encoding='utf-8-sig')
labels_bin = np.load('../data/labels_bin.npy')

with open('../data/label_cols.json', encoding='utf-8') as f:
    label_cols = json.load(f)

with open('../data/thresholds_kfold_finais.json', encoding='utf-8') as f:
    THRESHOLDS = json.load(f)

textos = df['nota_processada'].values

X_tv, X_test, Y_tv, Y_test = train_test_split(textos, labels_bin, test_size=0.20, random_state=SEED)
X_train, X_val, Y_train, Y_val = train_test_split(X_tv, Y_tv, test_size=0.10/0.80, random_state=SEED)
Y_true_test = Y_test.copy()

print(f"✅ Dados carregados | Teste: {len(X_test)} notas")

## 2. TF-IDF Baseline — Mesmo Holdout

In [ ]:
# NOTA: TF-IDF treinado nos mesmos dados de treino do BERT
# e avaliado no mesmo holdout (n=49) para comparação válida.

pipeline_tfidf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)),
    ('clf',   OneVsRestClassifier(LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)))
])
pipeline_tfidf.fit(X_train, Y_train)

probs_tfidf_val  = pipeline_tfidf.predict_proba(X_val)
probs_tfidf_test = pipeline_tfidf.predict_proba(X_test)

thresholds_tfidf = {}
for idx, label in enumerate(label_cols):
    melhor_t, melhor_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.05):
        preds = (probs_tfidf_val[:, idx] >= t).astype(int)
        f1    = f1_score(Y_val[:, idx], preds, zero_division=0)
        if f1 > melhor_f1:
            melhor_f1, melhor_t = f1, t
    thresholds_tfidf[label] = round(float(melhor_t), 2)

preds_tfidf_030 = (probs_tfidf_test >= 0.30).astype(int)
preds_tfidf_cal = np.zeros_like(probs_tfidf_test, dtype=int)
for idx, lbl in enumerate(label_cols):
    preds_tfidf_cal[:, idx] = (probs_tfidf_test[:, idx] >= thresholds_tfidf.get(lbl, 0.5)).astype(int)

print(f"✅ TF-IDF | holdout n={len(X_test)}")
print(f"   Threshold 0.30 — HL: {hamming_loss(Y_true_test, preds_tfidf_030):.4f}  F1-mac: {f1_score(Y_true_test, preds_tfidf_030, average='macro', zero_division=0):.4f}")
print(f"   Calibrado      — HL: {hamming_loss(Y_true_test, preds_tfidf_cal):.4f}  F1-mac: {f1_score(Y_true_test, preds_tfidf_cal, average='macro', zero_division=0):.4f}")

## 3. Probabilidades BERTimbau

In [ ]:
import sys
sys.path.insert(0, '../src')
from model import NurseXAIBERT
from train import NotasEnfermagemDataset

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizerFast.from_pretrained('neuralmind/bert-base-portuguese-cased')
ckpt      = torch.load('../outputs/nursexai_checkpoint.pt', map_location=device)
model     = NurseXAIBERT('neuralmind/bert-base-portuguese-cased', len(label_cols), 0.3).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def extrair_probs(textos_arr, labels_arr, max_len=256, bs=8):
    ds = NotasEnfermagemDataset(list(textos_arr), labels_arr, tokenizer, max_len)
    ld = DataLoader(ds, batch_size=bs, shuffle=False)
    pp, ll = [], []
    with torch.no_grad():
        for batch in ld:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            pp.append(torch.sigmoid(logits).cpu().numpy())
            ll.append(batch['labels'].numpy())
    return np.vstack(pp), np.vstack(ll)

probs_val,   _ = extrair_probs(X_val,   Y_val)
probs_teste, _ = extrair_probs(X_test,  Y_true_test)

preds_bert_030 = (probs_teste >= 0.30).astype(int)
preds_bert_cal = np.zeros_like(probs_teste, dtype=int)
for idx, lbl in enumerate(label_cols):
    preds_bert_cal[:, idx] = (probs_teste[:, idx] >= THRESHOLDS.get(lbl, 0.5)).astype(int)

hl_orig = hamming_loss(Y_true_test, preds_bert_030)
hl_cal  = hamming_loss(Y_true_test, preds_bert_cal)
print(f"✅ BERTimbau probabilidades extraídas")
print(f"   Threshold 0.30 — HL: {hl_orig:.4f}")
print(f"   Calibrado      — HL: {hl_cal:.4f}")
print(f"   Redução        : {(1-hl_cal/hl_orig)*100:.0f}%")

## 4. Bootstrap CI 95%

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n=1000):
    scores = []
    size   = len(y_true)
    np.random.seed(SEED)
    for _ in range(n):
        idx = resample(range(size), n_samples=size)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

f1_mac = partial(f1_score, average='macro', zero_division=0)
f1_mic = partial(f1_score, average='micro', zero_division=0)

MODELOS = {
    'TF-IDF  (threshold 0.30)' : preds_tfidf_030,
    'TF-IDF  (calibrado)'      : preds_tfidf_cal,
    'BERT    (threshold 0.30)' : preds_bert_030,
    'BERT    (calibrado) ★'    : preds_bert_cal,
}

print("=" * 88)
print(f"TABELA COMPARATIVA | Holdout n={len(X_test)} | Bootstrap CI 95% (n=1000)")
print("=" * 88)
print(f"\n{'Modelo':<32} {'F1-macro':>18} {'F1-micro':>18} {'Hamming Loss':>18}")
print("─" * 88)

resultados = {}
for nome, preds in MODELOS.items():
    m_mac,l_mac,u_mac = bootstrap_ci(Y_true_test, preds, f1_mac)
    m_mic,l_mic,u_mic = bootstrap_ci(Y_true_test, preds, f1_mic)
    m_hl, l_hl, u_hl  = bootstrap_ci(Y_true_test, preds, hamming_loss)
    resultados[nome]   = dict(f1_macro=m_mac,f1_macro_low=l_mac,f1_macro_high=u_mac,
                              f1_micro=m_mic,hamming_loss=m_hl,hl_low=l_hl,hl_high=u_hl)
    print(f"  {nome:<30} {m_mac:.3f} [{l_mac:.3f}–{u_mac:.3f}]  {m_mic:.3f} [{l_mic:.3f}–{u_mic:.3f}]  {m_hl:.3f} [{l_hl:.3f}–{u_hl:.3f}]")
print("─" * 88)

## 5. Relatório por Label

In [ ]:
print("── BERT calibrado ──")
print(classification_report(Y_true_test, preds_bert_cal,
      target_names=[l[:38] for l in label_cols], zero_division=0))

print("── TF-IDF calibrado ──")
print(classification_report(Y_true_test, preds_tfidf_cal,
      target_names=[l[:38] for l in label_cols], zero_division=0))

## 6. Guardar Resultados

In [ ]:
os.makedirs('../outputs', exist_ok=True)

pd.DataFrame(resultados).T.to_csv('../outputs/metrics_table.csv')

rows = []
for idx, lbl in enumerate(label_cols):
    f1b = f1_score(Y_true_test[:,idx], preds_bert_cal[:,idx],  zero_division=0)
    f1t = f1_score(Y_true_test[:,idx], preds_tfidf_cal[:,idx], zero_division=0)
    rows.append({'label':lbl,'f1_bert':round(f1b,3),'f1_tfidf':round(f1t,3),
                 'delta':round(f1b-f1t,3),'support':int(Y_true_test[:,idx].sum())})
pd.DataFrame(rows).sort_values('delta',ascending=False).to_csv('../outputs/per_label_results.csv',index=False)

np.save('../outputs/probs_val.npy',   probs_val)
np.save('../outputs/probs_teste.npy', probs_teste)

print("✅ Guardado em outputs/")
print("→ Avançar para: 04_lime_clix_analysis.ipynb")